<a href="https://colab.research.google.com/github/vneumannufprbr/TrabajosenPython/blob/main/Dashboard_Interactivo_Doble_Verificacion_con_Selector_Fechas_y_Graficos_Ok_vf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
#      *** DASHBOARD PARA ANALISIS DE DATOS DE DETECCIÓN DE ANOMALÍAS ***
# ==============================================================================
# Esta versión utiliza las mejores prácticas para dashboards interactivos
# en notebooks. Los mensajes de error de sincronización que puedan aparecer
# son inofensivos y un efecto secundario conocido de la comunicación interna
# de las librerías, pero no afectan la funcionalidad del dashboard.

# ------------------------------------------------------------------------------
# SECCIÓN 1: INSTALACIÓN E IMPORTACIÓN DE LIBRERÍAS
# ------------------------------------------------------------------------------
print("--- [Sección 1] Instalando e importando librerías...")
try:
    import sys
    !{sys.executable} -m pip install openpyxl plotly ipywidgets -q

    import pandas as pd
    import plotly.graph_objects as go
    import plotly.express as px
    import ipywidgets as widgets
    from IPython.display import display, clear_output, HTML
    from datetime import datetime, timedelta

    # Habilitación del gestor de widgets para Google Colab
    from google.colab import output
    output.enable_custom_widget_manager()

    print("✓ Librerías instaladas e importadas correctamente. Gestor de widgets habilitado.")
except Exception as e:
    print(f"Error en la Sección 1: {e}")

# ------------------------------------------------------------------------------
# SECCIÓN 2: CARGA Y PREPARACIÓN DE DATOS
# ------------------------------------------------------------------------------
print("\n--- [Sección 2] Cargando y preparando los datos...")
df = pd.DataFrame()
try:
    file_path = 'matriz_diagnostico_escalada_completa.xlsx'
    df = pd.read_excel(file_path)
    df['fecha'] = pd.to_datetime(df['fecha'])
    print("✓ Datos cargados y preparados correctamente.")
    print(f"  - Total de registros: {len(df)}")
    print(f"  - Rango de fechas: {df['fecha'].min().strftime('%Y-%m-%d')} a {df['fecha'].max().strftime('%Y-%m-%d')}")
except FileNotFoundError:
    print("✗ Error: El archivo 'matriz_diagnostico_escalada_completa.xlsx' no fue encontrado.")
except Exception as e:
    print(f"✗ Error inesperado al cargar los datos: {e}")

# PALETA DE COLORES ESTANDARIZADA
color_map = {
    'Anomalía Estructural': '#d32f2f', # Rojo
    'Anomalía Regional': '#ff9800',    # Naranja
    'Sensor Defectuoso': '#2196f3',    # Azul
    'Sistema Normal': '#4caf50'        # Verde
}

# ------------------------------------------------------------------------------
# SECCIÓN 3: DASHBOARD NIVEL 1 - PANEL DE CONTROL ESTRATÉGICO
# ------------------------------------------------------------------------------
if not df.empty:
    print("\n=============================================")
    print("  Panel de Control Estratégico (Nivel 1)")
    print("=============================================")

    # KPIs
    today = df['fecha'].max()
    kpi1_value = len(df[(df['fecha'] >= today - timedelta(days=7)) & (df['diagnostic_category'].isin(['Anomalía Estructural', 'Anomalía Regional']))])
    kpi2_value = df[(df['fecha'] >= today - timedelta(days=30)) & (df['diagnostic_category'] == 'Sensor Defectuoso')]['nombre_etiqueta'].nunique()
    health_df = df[df['fecha'] >= today - timedelta(days=30)]
    kpi3_value = (health_df['diagnostic_category'] == 'Sistema Normal').sum() / len(health_df) * 100 if not health_df.empty else 100
    print(f"\nKPI 1: Alertas Críticas (Últimos 7 días) -> {kpi1_value}")
    print(f"KPI 2: Sensores únicos 'Defectuoso' (Último mes) -> {kpi2_value}")
    print(f"KPI 3: Salud General del Sistema (Último mes) -> {kpi3_value:.2f}% Normal")
    print("\n--- Filtros del Panel de Control Estratégico ---")

    # Widgets de Control
    date_start_picker = widgets.DatePicker(description='Fecha Inicio:', style={'description_width': 'initial'}, value=df['fecha'].min().date())
    date_end_picker = widgets.DatePicker(description='Fecha Fin:', style={'description_width': 'initial'}, value=df['fecha'].max().date())
    update_button = widgets.Button(description='Analizar Selección del Nivel 1')

    # Widget de Leyenda
    legend_items = []
    for category, color in color_map.items():
        legend_items.append(
            f'<div style="display: flex; align-items: center; margin-right: 20px;">'
            f'<div style="width:15px; height:15px; background-color:{color}; margin-right:5px; border: 1px solid #555;"></div>'
            f'<span>{category}</span></div>'
        )
    legend_html = widgets.HTML(
        value=f'<div style="display: flex; flex-wrap: wrap; justify-content: center; margin-bottom: 15px;">{"".join(legend_items)}</div>'
    )

    # Creación de los FigureWidget
    fig_map = go.FigureWidget(layout=go.Layout(title="Estado Más Crítico por Local Geológico", margin=dict(t=50, l=25, r=25, b=25)))
    fig_rank = go.FigureWidget(layout=go.Layout(
        title="Top 10 Grupos con Más Anomalías Críticas (Histórico)",
        yaxis={'categoryorder':'total ascending'},
        showlegend=False
    ))

    def update_level1_charts(b):
        start_date = pd.to_datetime(date_start_picker.value)
        end_date = pd.to_datetime(date_end_picker.value)

        with fig_map.batch_update():
            fig_map.data = []
            filtered_df_l1 = df[(df['fecha'] >= start_date) & (df['fecha'] <= end_date)].copy()
            if not filtered_df_l1.empty:
                category_severity = {'Anomalía Estructural': 4, 'Anomalía Regional': 3, 'Sensor Defectuoso': 2, 'Sistema Normal': 1}
                filtered_df_l1['severity'] = filtered_df_l1['diagnostic_category'].map(category_severity)
                structure_status = filtered_df_l1.loc[filtered_df_l1.groupby('local_geologico')['severity'].idxmax()]
                structure_status['worst_category'] = structure_status['severity'].map({v: k for k, v in category_severity.items()})
                new_treemap_trace = px.treemap(
                    structure_status, path=[px.Constant("Local Geológico"), 'local_geologico'],
                    color='worst_category', color_discrete_map=color_map
                ).data[0]
                fig_map.add_trace(new_treemap_trace)
                fig_map.layout.title = f'Estado Más Crítico ({start_date.strftime("%Y-%m-%d")} a {end_date.strftime("%Y-%m-%d")})'
            else:
                fig_map.layout.title = f'Sin datos entre {start_date.strftime("%Y-%m-%d")} y {end_date.strftime("%Y-%m-%d")}'

        with fig_rank.batch_update():
            fig_rank.data = []
            critical_df = df[df['diagnostic_category'].isin(['Anomalía Estructural', 'Anomalía Regional'])]
            ranking = critical_df['etiqueta_padre'].value_counts().nlargest(10).sort_values()
            fig_rank.add_bar(x=ranking.values, y=ranking.index, orientation='h', marker_color='#c62828')

    update_button.on_click(update_level1_charts)
    update_level1_charts(None)

    # Visualización
    controls_l1 = widgets.VBox([widgets.HBox([date_start_picker, date_end_picker]), update_button])
    display(controls_l1, legend_html, fig_map, fig_rank)


# ------------------------------------------------------------------------------
# SECCIÓN 4: DASHBOARD NIVEL 2 - ANÁLISIS DETALLADO
# ------------------------------------------------------------------------------
if not df.empty:
    print("\n=============================================")
    print("    Análisis Detallado por Grupo (Nivel 2)")
    print("=============================================")

    # --- WIDGETS PARA NIVEL 2 ---
    grupos_disponibles = ['Todos los Grupos'] + sorted(df['etiqueta_padre'].dropna().unique().tolist())
    parent_label_dropdown = widgets.Dropdown(options=grupos_disponibles, description='Grupo:', style={'description_width': 'initial'})

    categorias_disponibles = ['Todas'] + sorted(df['diagnostic_category'].dropna().unique().tolist())
    category_dropdown_l2 = widgets.Dropdown(options=categorias_disponibles, value='Todas', description='Categoría:', style={'description_width': 'initial'})

    analyze_button_l2 = widgets.Button(description='Analizar Selección del Nivel 2')

    output_table_l2 = widgets.Output()
    fig_ts = go.FigureWidget(layout=go.Layout(title="Seleccione sus filtros y haga clic en analizar"))

    def analyze_selection(b):
        # 1. Leer valores de TODOS los filtros
        start_date_l1 = pd.to_datetime(date_start_picker.value)
        end_date_l1 = pd.to_datetime(date_end_picker.value)
        parent_label = parent_label_dropdown.value
        selected_category = category_dropdown_l2.value

        # 2. Aplicar TODOS los filtros a los datos
        filtered_df_l2 = df[
            (df['fecha'] >= start_date_l1) &
            (df['fecha'] <= end_date_l1)
        ].copy()

        if parent_label != 'Todos los Grupos':
            filtered_df_l2 = filtered_df_l2[filtered_df_l2['etiqueta_padre'] == parent_label]

        if selected_category != 'Todas':
            filtered_df_l2 = filtered_df_l2[filtered_df_l2['diagnostic_category'] == selected_category]

        # 3. Actualizar la tabla
        with output_table_l2:
            clear_output(wait=True)
            print(f"\n--- Análisis para Grupo: '{parent_label}' | Categoría: '{selected_category}' ---")

            record_count = len(filtered_df_l2)

            if record_count == 0:
                print("No se encontraron registros que coincidan con la selección.")
            else:
                # *** MEJORA: Mensaje dinámico y límite de 100 líneas ***
                display_message = (
                    f"<b>Mostrando los primeros 100 de {record_count} registros históricos encontrados:</b>"
                    if record_count > 100
                    else f"<b>Registros Históricos Encontrados: {record_count}</b>"
                )
                display(HTML(display_message))

                columnas_a_mostrar = ['fecha', 'nombre_etiqueta', 'valor_promedio', 'diagnostic_category', 'local_geologico']
                df_to_display = filtered_df_l2[[col for col in columnas_a_mostrar if col in filtered_df_l2.columns]].head(100)
                display(df_to_display.style.hide(axis="index"))

        # 4. Actualizar el gráfico
        with fig_ts.batch_update():
            fig_ts.data = []
            fig_ts.layout.title = f'Serie de Tiempo para Grupo: {parent_label} ({start_date_l1.strftime("%Y-%m-%d")} a {end_date_l1.strftime("%Y-%m-%d")})'
            if not filtered_df_l2.empty:
                num_sensores = filtered_df_l2['nombre_etiqueta'].nunique()
                if num_sensores > 50:
                     with output_table_l2: # Re-usa el output de la tabla para mostrar la advertencia
                        print(f"\nADVERTENCIA: Se encontraron {num_sensores} sensores. El gráfico puede tardar en cargar o ser ilegible.")

                # Para el gráfico, también es prudente limitar la cantidad de datos a graficar
                df_for_plot = filtered_df_l2.head(2000) # Límite de ejemplo para graficar

                for sensor_name in df_for_plot['nombre_etiqueta'].unique():
                    sensor_df = df_for_plot[df_for_plot['nombre_etiqueta'] == sensor_name]
                    fig_ts.add_trace(go.Scatter(x=sensor_df['fecha'], y=sensor_df['valor_promedio'], name=sensor_name, mode='lines'))

                anomalies_df = df_for_plot[df_for_plot['diagnostic_category'] != 'Sistema Normal']
                if not anomalies_df.empty:
                    fig_ts.add_trace(go.Scatter(x=anomalies_df['fecha'], y=anomalies_df['valor_promedio'], mode='markers',
                                                marker=dict(color=anomalies_df['diagnostic_category'].map(color_map), size=10, symbol='x'),
                                                name='Anomalía', hovertext=anomalies_df['diagnostic_category']))

    analyze_button_l2.on_click(analyze_selection)

    # Visualización de los controles y las salidas del Nivel 2
    controls_l2 = widgets.VBox([widgets.HBox([parent_label_dropdown, category_dropdown_l2]), analyze_button_l2])
    display(controls_l2, output_table_l2, fig_ts)

else:
    print("\n--- No se cargaron datos. Los dashboards no pueden ser mostrados. ---")

--- [Sección 1] Instalando e importando librerías...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 26.3 MB/s eta 0:00:00
✓ Librerías instaladas e importadas correctamente. Gestor de widgets habilitado.

--- [Sección 2] Cargando y preparando los datos...
✓ Datos cargados y preparados correctamente.
  - Total de registros: 75566
  - Rango de fechas: 1981-06-01 a 2022-11-18

  Panel de Control Estratégico (Nivel 1)

KPI 1: Alertas Críticas (Últimos 7 días) -> 2
KPI 2: Sensores únicos 'Defectuoso' (Último mes) -> 0
KPI 3: Salud General del Sistema (Último mes) -> 0.00% Normal

--- Filtros del Panel de Control Estratégico ---


HTML(value='<div style="display: flex; flex-wrap: wrap; justify-content: center; margin-bottom: 15px;"><div st…

FigureWidget({
    'data': [{'branchvalues': 'total',
              'customdata': array([['Anomalía Estructural'],
                                   ['Anomalía Estructural'],
                                   ['Anomalía Estructural'],
                                   ['Anomalía Estructural'],
                                   ['Anomalía Estructural'],
                                   ['Anomalía Estructural'],
                                   ['Anomalía Regional'],
                                   ['Anomalía Regional'],
                                   ['Anomalía Regional'],
                                   ['(?)']], dtype=object),
              'domain': {'x': [0.0, 1.0], 'y': [0.0, 1.0]},
              'hovertemplate': ('labels=%{label}<br>count=%{val' ... '{customdata[0]}<extra></extra>'),
              'ids': array(['Local Geológico/CONTATO C/D', 'Local Geológico/CONTATO CONCRETO/ROCHA',
                            'Local Geológico/CONTATO D/E', 'Local Geológico/JUNTA

FigureWidget({
    'data': [{'marker': {'color': '#c62828'},
              'orientation': 'h',
              'type': 'bar',
              'uid': '78b8527f-7d99-47bd-adc0-0d9a761d84c3',
              'x': array([  4,   4,   4,   4,   6,   8,  44,  88, 132, 132]),
              'y': array(['PD-E-006/1', 'PD-E-006/2', 'PS-E-024', 'PS-E-023', 'EM-E-003',
                          'PS-E-021', 'EM-E-001', 'PD-E-006/3', 'EM-E-004', 'EM-E-002'],
                         dtype=object)}],
    'layout': {'showlegend': False,
               'template': '...',
               'title': {'text': 'Top 10 Grupos con Más Anomalías Críticas (Histórico)'},
               'yaxis': {'categoryorder': 'total ascending'}}
})


    Análisis Detallado por Grupo (Nivel 2)


...

KeyError: 1

KeyError: 1

KeyError: 1

KeyError: 1

KeyError: 1